<div align="center">
  <h3><b>ESCUELA POLITÉCNICA NACIONAL</b></h3>
  <h3><b>FACULTAD DE INGENIERÍA EN SISTEMAS</b></h3>
  <h3><b>INGENIERÍA EN CIENCIAS DE LA COMPUTACIÓN</b></h3>
  <h3><b>RECUPERACIÓN DE LA INFORMACIÓN</b></h3>
</div>

---
**Nombre**   Mark Hernández        
**Fecha**    08/07/26  
**Docente**  Iván Carrera

# Ejercicio: Web Scraping

## Objetivo de la práctica

El objetivo de este ejercicio es construir un web scraper que recoja datos de un website.

### Parte 0: Planificar
1. Identificar los datos que quieres obtener.
2. Elegir el sitio web objetivo.
3. Planificar la estructura del corpus.

## Parte 1: Entender el sitio web objetivo

- Analizar la estructura de la página web a ser analizada.
- Identificar los elementos HTML que contienen los datos buscados.

In [1]:
from bs4 import BeautifulSoup

file ='./rotisserie-chicken.html'

# Load the HTML file
with open(file, "r", encoding="utf-8") as file:
    html_content = file.read()
    
# Parse the HTML content with BeautifulSoup
soup = BeautifulSoup(html_content, "html.parser")

In [2]:
# Extracting the recipe title
title = soup.find("meta", {"property": "og:title"})["content"]
title

'Rotisserie Chicken'

In [3]:
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
for ingredient in ingredients_section:
    print(ingredient.text.strip())

1 (3 pound) whole chicken
1 pinch salt
¼ cup butter, melted
1 tablespoon salt
1 tablespoon ground paprika
¼ tablespoon ground black pepper


## Parte 2: Obtener los datos deseados

* Buscar dentro del contenido HTML y extraer la información.

In [4]:
# Extracting the description
description = soup.find("meta", {"name": "description"})["content"]

# Extracting the ingredients
ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]

# Extracting the instructions
instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
instructions = [instruction.get_text().strip() for instruction in instructions_section]

# Extracting the nutrition information
nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]

# Print the extracted information
print("Recipe Title:", title)
print("Description:", description)
print("Ingredients:")
for ingredient in ingredients:
    print("-", ingredient)
print("Instructions:")
for i, instruction in enumerate(instructions, 1):
    print(f"{i}. {instruction}")
print("Nutrition Facts:")
for fact in nutrition_facts:
    print("-", fact)


Recipe Title: Rotisserie Chicken
Description: Rotisserie chicken that's easy to cook on a gas grill and turns out moist and juicy with crispy skin. This is a simple recipe that our family loves.
Ingredients:
- 1 (3 pound) whole chicken
- 1 pinch salt
- ¼ cup butter, melted
- 1 tablespoon salt
- 1 tablespoon ground paprika
- ¼ tablespoon ground black pepper
Instructions:
1. Intimidated by the idea of making a rotisserie chicken at home? We're here to help. Get your grill and rotisserie attachment ready — you'll want to try this recipe ASAP.
2. Here's what you'll need to make rotisserie chicken at home:
3. · Whole Chicken: This recipe is meant for a whole 3-pound chicken. If your chicken is larger or smaller, you'll have to adjust the cooking time.· Butter: Butter keeps the chicken moist and juicy, while giving the seasonings something to stick to.· Seasonings: The rotisserie chicken is simply seasoned with salt, pepper, and paprika.
4. You'll find the full, step-by-step recipe below — b

## Parte 3: Obtener enlaces relacionados
* Encontrar links a otras recetas para completar el corpus

In [5]:
# Find all the links to other recipes
recipe_links = soup.find_all("a", href=True)

# Filter and print only the links that are likely to be recipes
recipe_urls = set()
for link in recipe_links:
    href = link['href']
    if "/recipe/" in href:
        recipe_urls.add(href)

# Convert the set to a list for easier handling in Dataframe
recipe_urls = list(recipe_urls)  

# Print the recipe URLs
print(f"Recetas únicas encontradas: {len(recipe_urls)}\n")
print("Linked Recipes:")
for url in recipe_urls:
    print(url)

Recetas únicas encontradas: 16

Linked Recipes:
https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
https://www.allrecipes.com/recipe/258659/rosemary-buttermilk-chicken/
https://www.allrecipes.com/recipe/228070/the-best-beer-can-chicken-ever/
https://www.allrecipes.com/recipe/238575/cilantro-lime-grilled-chicken/
https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
https://www.allrecipes.com/recipe/214619/bbq-beer-can-chicken/
https://www.allrecipes.com/recipe/281255/smoked-whole-chicken/
https://www.allrecipes.com/recipe/8998/darn-good-chicken/
https://www.allrecipes.com/recipe/214618/beer-can-chicken/
https://www.allrecipes.com/recipe/221093/good-frickin-paprika-chicken/
https://www.allrecipes.com/recipe/19944/drunk-chicken/
https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/
https://www.allrecipes.com/recipe/34957

Con estos pasos bien realizados pasamos a la construcción del corpus para las 16 recetas.

In [6]:
import requests
import pandas as pd
import time
import random

# 1. Definimos las cabeceras para simular un navegador web (Chrome en Windows)
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.5",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive",
    "Upgrade-Insecure-Requests": "1"
}

content_for_rag = f"Receta: {title}. Ingredientes: {ingredients}. Preparación: {instructions}. Nutrición: {nutrition_facts}"

# Lista para almacenar los diccionarios con la información de cada receta
corpus_data = [{
    "url": "https://www.allrecipes.com/recipe/30522/rotisserie-chicken/",
    "titulo": title,
    "ingredientes": ingredients,
    "instrucciones": instructions,
    "texto_completo": content_for_rag  # Esta columna será la base para los embeddings
}]

print("Receta Rostisserie Chicken agregada correctamente al corpus.")
print("Iniciando la extracción del corpus mediante Scraping...")

for url in recipe_urls:
    try:
        # Realizar la petición a la página de la receta
        response = requests.get(url, headers=headers)
        
        if response.status_code == 200:
            soup = BeautifulSoup(response.content, "html.parser")
            
            # 1. Extraer el Título
            title = soup.find("meta", {"property": "og:title"})["content"]
            
            # 2. Extraer los Ingredientes
            ingredients_section = soup.find_all("li", class_="mm-recipes-structured-ingredients__list-item")
            ingredients = [ingredient.get_text().strip() for ingredient in ingredients_section]
            
            # 3. Extraer las Instrucciones
            instructions_section = soup.find_all("p", class_="comp mntl-sc-block mntl-sc-block-html")
            instructions = [instruction.get_text().strip() for instruction in instructions_section]

            # 4. Extraer la Nutrición
            nutrition_section = soup.find_all("span", class_="mm-recipes-nutrition-facts-label__nutrient-name mm-recipes-nutrition-facts-label__nutrient-name--has-postfix")
            nutrition_facts = [fact.parent.get_text().strip().replace('\n', ' ') for fact in nutrition_section]
            
            # Juntar todo el contenido en un solo campo de "documento" para el RAG, 
            # o mantenerlo separado según prefieras.
            ingredients_text = " | ".join(ingredients)
            content_for_rag = f"Receta: {title}. Ingredientes: {ingredients_text}. Preparación: {instructions}. Nutrición: {', '.join(nutrition_facts)}"
            
            # Añadir los datos a nuestra lista
            corpus_data.append({
                "url": url,
                "titulo": title,
                "ingredientes": ingredients_text,
                "instrucciones": instructions,
                "texto_completo": content_for_rag # Esta columna será la base para los embeddings
            })
            
            print(f"Extraída: {title}")
            
        else:
            print(f"Error HTTP {response.status_code} al acceder a {url}")
            
    except Exception as e:
        print(f"Error al procesar la URL {url}: {e}")
        
    # Pausa de 1 segundo entre peticiones por cortesía
    time.sleep(random.uniform(1.5, 3.0))

# Convertir la lista de diccionarios en un DataFrame de pandas
df_corpus = pd.DataFrame(corpus_data)

print(f"\nExtracción completada. DataFrame creado con {len(df_corpus)} registros.")

# Mostrar las primeras filas para verificar
df_corpus.head(17)

Receta Rostisserie Chicken agregada correctamente al corpus.
Iniciando la extracción del corpus mediante Scraping...
Extraída: Beer Butt Chicken
Extraída: Rosemary Buttermilk Chicken
Extraída: The Best Beer Can Chicken Ever
Extraída: Cilantro-Lime Grilled Chicken
Extraída: Smoked Beer Butt Chicken
Extraída: Buttermilk Barbecue Chicken
Extraída: Miso Honey Chicken
Extraída: Best Beer Can Chicken
Extraída: Smoked Whole Chicken
Extraída: Darn Good Chicken
Extraída: Beer Can Chicken
Extraída: Good Frickin’ Paprika Chicken
Extraída: Drunk Chicken
Extraída: Grilled Chicken Under a Brick
Extraída: Easy Barbeque Chicken
Extraída: Grilled Spatchcocked Chicken

Extracción completada. DataFrame creado con 17 registros.


,url,titulo,ingredientes,instrucciones,texto_completo
0,https://www.allrecipes.com/recipe/30522/rotiss...,Rotisserie Chicken,"[1 (3 pound) whole chicken, 1 pinch salt, ¼ cu...",[Intimidated by the idea of making a rotisseri...,Receta: Rotisserie Chicken. Ingredientes: ['1 ...
1,https://www.allrecipes.com/recipe/14531/beer-b...,Beer Butt Chicken,"1 cup butter, divided | 2 tablespoons garlic s...",[Preheat an outdoor grill for low heat and lig...,Receta: Beer Butt Chicken. Ingredientes: 1 cup...
2,https://www.allrecipes.com/recipe/258659/rosem...,Rosemary Buttermilk Chicken,"2 ½ cups buttermilk | 15 cloves garlic, minced...","[Mix buttermilk, garlic, paprika, salt, and pe...",Receta: Rosemary Buttermilk Chicken. Ingredien...
3,https://www.allrecipes.com/recipe/228070/the-b...,The Best Beer Can Chicken Ever,1 cup chocolate stout beer | 3 green Thai chi...,[Preheat grill for medium heat. If using charc...,Receta: The Best Beer Can Chicken Ever. Ingred...
4,https://www.allrecipes.com/recipe/238575/cilan...,Cilantro-Lime Grilled Chicken,"½ cup chopped fresh cilantro | 4 limes, juice...","[Whisk cilantro, lime juice, garlic salt, and ...",Receta: Cilantro-Lime Grilled Chicken. Ingredi...
5,https://www.allrecipes.com/recipe/222936/smoke...,Smoked Beer Butt Chicken,1 (3 pound) whole chicken | ¼ cup vegetable oi...,[Preheat grill for medium heat and lightly oil...,Receta: Smoked Beer Butt Chicken. Ingredientes...
6,https://www.allrecipes.com/recipe/275062/butte...,Buttermilk Barbecue Chicken,2 cups buttermilk | ¼ cup brown sugar | 1 tabl...,"[Whisk buttermilk, brown sugar, cider vinegar,...",Receta: Buttermilk Barbecue Chicken. Ingredien...
7,https://www.allrecipes.com/recipe/264278/miso-...,Miso Honey Chicken,3 tablespoons white miso | 2 tablespoons honey...,[Combine miso and honey in a bowl. Pour in ric...,Receta: Miso Honey Chicken. Ingredientes: 3 ta...
8,https://www.allrecipes.com/recipe/214619/bbq-b...,Best Beer Can Chicken,2 cups cherry wood chips | ½ cup dark brown su...,[Soak wood chips in water for at least 1 hour....,Receta: Best Beer Can Chicken. Ingredientes: 2...
9,https://www.allrecipes.com/recipe/281255/smoke...,Smoked Whole Chicken,2 tablespoons paprika | 2 tablespoons chili po...,[Preheat a smoker to 250 degrees F (120 degree...,Receta: Smoked Whole Chicken. Ingredientes: 2 ...


## Parte 4: Hacer RAG con las recetas obtenidas
* Una vez que se ha construido el corpus, implementar y desplegar RAG para realizar búsquedas en el corpus

#### Indexación y Bases de datos Vectoriales

In [16]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# 1. Cargar el modelo que transformará texto en embeddings (vectores)
print("Cargando el modelo de embeddings...")
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# 2. Extraer todos los textos del DataFrame que armamos antes
documentos_corpus = df_corpus['texto_completo'].tolist()

# 3. Transformar nuestro texto en números (Esto puede tomar unos segundos)
print("Generando vectores para las recetas...")
embeddings_recetas = modelo_embeddings.encode(documentos_corpus)

# 4. Crear el Índice FAISS (nuestra "Base de Datos Vectorial")
# FAISS necesita saber el tamaño exacto del vector (la cantidad de dimensiones)
dimensiones = embeddings_recetas.shape[1]
indice_vectorial = faiss.IndexFlatL2(dimensiones) # Utilizamos la distancia euclidiana (L2) para medir cercanía

# 5. Guardar los vectores en el índice
indice_vectorial.add(np.array(embeddings_recetas))

print(f"¡Éxito! Índice vectorial creado. Total de recetas indexadas: {indice_vectorial.ntotal}")

Cargando el modelo de embeddings...


c:\Users\mark_\Documents\ir26a\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\mark_\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3715.99it/s]


Generando vectores para las recetas...
¡Éxito! Índice vectorial creado. Total de recetas indexadas: 17


### Recuperación

Generamos una funció capaz de buscar el top k de documentos recuperados basados en la Query realizada.

In [17]:
# Definimos la función de recuperación
def buscar_recetas(query, k=3):
    """
    Busca las 'k' recetas más relevantes para una consulta dada.
    """
    # 1. Convertir la pregunta del usuario en un vector (embedding)
    # Es crucial usar el mismo modelo que usamos para el corpus
    query_vector = modelo_embeddings.encode([query])
    
    # 2. Buscar en FAISS los 'k' vectores más cercanos
    # FAISS devuelve dos cosas: 
    #   - D: Un arreglo con las distancias (qué tan similares son)
    #   - I: Un arreglo con los índices (la posición de la fila en nuestro DataFrame)
    distancias, indices = indice_vectorial.search(np.array(query_vector), k)
    
    # 3. Recuperar los textos originales usando los índices
    resultados = []
    
    # Iteramos sobre los índices encontrados
    for idx in indices[0]:
        if idx != -1: # -1 es el código de FAISS si no encuentra suficientes resultados
            # Extraemos la fila completa de nuestro DataFrame
            receta_encontrada = df_corpus.iloc[idx]
            
            # Guardamos la información útil en un diccionario
            resultados.append({
                "titulo": receta_encontrada["titulo"],
                "url": receta_encontrada["url"],
                "texto_completo": receta_encontrada["texto_completo"]
            })
            
    return resultados

Probamos nuestra función con tres ejemplos distintos de prueba.

#### PRUEBA 1

In [18]:
pregunta_usuario = "Quiero preparar algo con pollo que sea al horno"
print(f"Pregunta: '{pregunta_usuario}'\n")

# Llamamos a la función pidiendo los 2 mejores resultados
recetas_relevantes = buscar_recetas(pregunta_usuario, k=3)

# Imprimimos los resultados para verificar que la búsqueda funciona
for i, receta in enumerate(recetas_relevantes, 1):
    print(f"--- Resultado {i} ---")
    print(f"Título: {receta['titulo']}")
    print(f"Enlace: {receta['url']}")
    # Imprimimos solo los primeros 200 caracteres del texto para no saturar la pantalla
    print(f"Contexto recuperado: {receta['texto_completo'][:200]}...\n")

Pregunta: 'Quiero preparar algo con pollo que sea al horno'

--- Resultado 1 ---
Título: Grilled Chicken Under a Brick
Enlace: https://www.allrecipes.com/recipe/275044/grilled-chicken-under-a-brick/
Contexto recuperado: Receta: Grilled Chicken Under a Brick. Ingredientes: 1 (3 pound) whole chicken | 1 teaspoon olive oil | 1 pinch salt and freshly ground black pepper to taste | heavy duty aluminum foil | 2 clay bricks...

--- Resultado 2 ---
Título: Darn Good Chicken
Enlace: https://www.allrecipes.com/recipe/8998/darn-good-chicken/
Contexto recuperado: Receta: Darn Good Chicken. Ingredientes: 1 (2 to 3 pound) whole chicken, cut into pieces | ½ cup honey, warmed slightly | ½ cup prepared yellow mustard | ½ teaspoon ground nutmeg. Preparación: ['In a ...

--- Resultado 3 ---
Título: Grilled Spatchcocked Chicken
Enlace: https://www.allrecipes.com/recipe/274724/grilled-spatchcocked-chicken/
Contexto recuperado: Receta: Grilled Spatchcocked Chicken. Ingredientes: ¼ cup kosher salt | water | 

#### PRUEBA 2

In [19]:
# --- Búsqueda conceptual y nutricional ---

pregunta_concepto = "Una comida ligera, saludable y baja en calorías"
print(f"Prueba Conceptual: '{pregunta_concepto}'\n")

resultados_concepto = buscar_recetas(pregunta_concepto, k=3)

for i, receta in enumerate(resultados_concepto, 1):
    print(f"--- Resultado {i} ---")
    print(f"Título: {receta['titulo']}")
    print(f"Enlace: {receta['url']}")
    print(f"Contexto recuperado: {receta['texto_completo'][:250]}...\n")

Prueba Conceptual: 'Una comida ligera, saludable y baja en calorías'

--- Resultado 1 ---
Título: Miso Honey Chicken
Enlace: https://www.allrecipes.com/recipe/264278/miso-honey-chicken/
Contexto recuperado: Receta: Miso Honey Chicken. Ingredientes: 3 tablespoons white miso | 2 tablespoons honey | ¼ cup rice vinegar | 2 teaspoons hot sauce | 1 tablespoon kosher salt | 1  whole chicken, halved, wing tips separated | kosher salt to taste | ½  lemon, cut in...

--- Resultado 2 ---
Título: Beer Butt Chicken
Enlace: https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
Contexto recuperado: Receta: Beer Butt Chicken. Ingredientes: 1 cup butter, divided | 2 tablespoons garlic salt, divided | 2 tablespoons paprika, divided | salt and pepper to taste | 1 (12 fluid ounce) can beer | 1 (4 pound) whole chicken. Preparación: ['Preheat an outdo...

--- Resultado 3 ---
Título: Buttermilk Barbecue Chicken
Enlace: https://www.allrecipes.com/recipe/275062/buttermilk-barbecue-chicken/
Contexto recupe

#### PRUEBA 3

In [20]:
# --- PRUEBA 3: Búsqueda por ocasión y tipo de plato ---

pregunta_ocasion = "Un postre muy dulce con chocolate para el fin de semana"
print(f"Prueba de Ocasión: '{pregunta_ocasion}'\n")

resultados_ocasion = buscar_recetas(pregunta_ocasion, k=3)

for i, receta in enumerate(resultados_ocasion, 1):
    print(f"--- Resultado {i} ---")
    print(f"Título: {receta['titulo']}")
    print(f"Enlace: {receta['url']}")
    print(f"Contexto recuperado: {receta['texto_completo'][:250]}...\n")

Prueba de Ocasión: 'Un postre muy dulce con chocolate para el fin de semana'

--- Resultado 1 ---
Título: Beer Butt Chicken
Enlace: https://www.allrecipes.com/recipe/14531/beer-butt-chicken/
Contexto recuperado: Receta: Beer Butt Chicken. Ingredientes: 1 cup butter, divided | 2 tablespoons garlic salt, divided | 2 tablespoons paprika, divided | salt and pepper to taste | 1 (12 fluid ounce) can beer | 1 (4 pound) whole chicken. Preparación: ['Preheat an outdo...

--- Resultado 2 ---
Título: Smoked Beer Butt Chicken
Enlace: https://www.allrecipes.com/recipe/222936/smoked-beer-butt-chicken/
Contexto recuperado: Receta: Smoked Beer Butt Chicken. Ingredientes: 1 (3 pound) whole chicken | ¼ cup vegetable oil | 3 tablespoons liquid smoke flavoring | 1 clove garlic, peeled | ⅔ (12 ounce) can beer | sea salt to taste | ground black pepper to taste | 1 pinch garli...

--- Resultado 3 ---
Título: The Best Beer Can Chicken Ever
Enlace: https://www.allrecipes.com/recipe/228070/the-best-beer-can-chi

### Generación

Creamos una función para realizar el RAG con el modelo LLM.

In [25]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

print("Cargando el Tokenizador y el Modelo de Lenguaje (LLM)...")
# Descargamos e instanciamos explícitamente los componentes
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
modelo_llm = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

def responder_con_rag(pregunta):
    """
    Pipeline completo de RAG: Recuperación + Generación explícita.
    """
    print(f"Pregunta: {pregunta}")
    print("Buscando en la base de datos vectorial...")
    
    # 1. Recuperación
    documentos_recuperados = buscar_recetas(pregunta, k=3)
    
    if not documentos_recuperados:
        return "No encontré recetas relevantes para tu búsqueda."
    
    # 2. Construir el Contexto
    contexto_texto = ""
    for i, doc in enumerate(documentos_recuperados):
        contexto_texto += f"Recipe {i+1} ({doc['titulo']}): {doc['texto_completo']}\n\n"
        
    # 3. Ingeniería de Prompts (Evaluando idioma cruzado)
    prompt = f"""
    Answer the following question in SPANISH based ONLY on the provided context. 
    If the answer is not contained in the context, you must reply exactly with: "No puedo responder esto basándome en las recetas proporcionadas."
    
    Context:
    {contexto_texto}
    
    Question: {pregunta}
    
    Answer in Spanish:
    """
    
    print("Generando respuesta con el LLM...\n")
    
    print("Generando respuesta con el LLM...\n")
    
    # 4. Generación explícita
    # Convertimos el texto del prompt a tensores que el modelo entiende
    inputs = tokenizer(prompt, return_tensors="pt")
    
    # Generamos la respuesta. do_sample=True permite usar la temperatura.
    outputs = modelo_llm.generate(
        **inputs, 
        max_new_tokens=150, 
        temperature=0.3,
        do_sample=True
    )
    
    # Decodificamos los tensores de vuelta a texto humano
    respuesta_generada = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # 5. Estructurar la salida
    fuentes = [doc['titulo'] for doc in documentos_recuperados]
    
    resultado_final = {
        "respuesta": respuesta_generada,
        "fuentes_utilizadas": fuentes
    }
    
    return resultado_final

Cargando el Tokenizador y el Modelo de Lenguaje (LLM)...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1884.18it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Realizamos las pruebas adecuadas para verificar el correcto funcionamiento del RAG.

In [26]:
# Prueba 1: Extracción de datos específicos (Tiempo, temperatura o cantidades)
# Evalúa si el LLM puede localizar un dato numérico exacto dentro del texto recuperado.
prueba_1 = "At what exact temperature should the oven be to cook the chicken, and how long does it take?"

# Prueba 2: Síntesis de instrucciones
# Evalúa si el modelo puede leer varios pasos y resumir el proceso.
prueba_2 = "What are the main steps to prepare the dressing or spice rub for the recipe?"

# Prueba 3: Resistencia a Alucinaciones (Prueba Negativa)
# El modelo debe reconocer que esta información NO está en el corpus.
prueba_3 = "How do I prepare a Hawaiian pizza with pineapple?"

lista_pruebas = [prueba_1, prueba_2, prueba_3]

for i, pregunta_prueba in enumerate(lista_pruebas, 1):
    print(f"\n{'='*50}")
    print(f"🔄 EJECUTANDO PRUEBA {i}...")
    
    # Llamamos a tu función actualizada
    resultado = responder_con_rag(pregunta_prueba)
    
    print(f"\n🤖 RESPUESTA DE LA IA:\n{resultado['respuesta']}")
    print("\n📚 FUENTES (Contexto utilizado):")
    for fuente in resultado['fuentes_utilizadas']:
        print(f"- {fuente}")
print(f"{'='*50}")


🔄 EJECUTANDO PRUEBA 1...
Pregunta: At what exact temperature should the oven be to cook the chicken, and how long does it take?
Buscando en la base de datos vectorial...


[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1482 > 512). Running this sequence through the model will result in indexing errors


Generando respuesta con el LLM...

Generando respuesta con el LLM...


🤖 RESPUESTA DE LA IA:
Cuál es la temperatura exacta del aire y cuánto es la vez que se llevará a prehearse el cárcere?

📚 FUENTES (Contexto utilizado):
- Darn Good Chicken
- Grilled Spatchcocked Chicken
- Cilantro-Lime Grilled Chicken

🔄 EJECUTANDO PRUEBA 2...
Pregunta: What are the main steps to prepare the dressing or spice rub for the recipe?
Buscando en la base de datos vectorial...
Generando respuesta con el LLM...

Generando respuesta con el LLM...


🤖 RESPUESTA DE LA IA:
['Mixes buttermilk, garlic, paprika, salt, y pepper en una granja; stir to combine.', 'Massage sprigs to release fragrant oils. Put rosemary in a large resealable plastic bag; pour in buttermilk mixture. Add chicken parts to the bag, seal carefully, and place in refrigerator. Marinate, turning the bag occasionally, 4 hours to overnight.', 'Preheat griddle para alta velocidad y lquido el grate.', 'Season el cálculo con salt y 

📚 FUENTES (Cont